# 01. Загрузка исторических данных ОФЗ с Московской биржи

Ноутбук скачивает дневные исторические данные по выбранным выпускам ОФЗ через ISS API Московской биржи.
Каждая облигация сохраняется в отдельный CSV-файл, дополнительно формируется общий объединённый файл.

Выходные файлы сохраняются в `data/raw/ofz_moex_csv_2016_2026/`.


## 1. Импорт библиотек и пути проекта


In [1]:
import time
from pathlib import Path

import requests
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SUMMARY_DIR = DATA_DIR / "summary"
FIGURES_DIR = PROJECT_ROOT / "report" / "figures"

for path in [RAW_DIR, PROCESSED_DIR, SUMMARY_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data dir:", RAW_DIR.resolve())
print("Processed data dir:", PROCESSED_DIR.resolve())


Project root: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only
Raw data dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw
Processed data dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed


## 2. Параметры загрузки и список выпусков


In [3]:
START_DATE = "2016-05-03"
END_DATE = "2026-05-03"
BOARD = "TQOB"

OUT_DIR = RAW_DIR / "ofz_moex_csv_2016_2026"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BONDS = [
    # Краткосрочные ОФЗ-ПД
    {
        "secid": "SU26219RMFS4",
        "issue_name": "ОФЗ-ПД 26219",
        "coupon_type": "fixed",
        "analysis_group": "short_fixed_selected",
    },
    {
        "secid": "SU26226RMFS9",
        "issue_name": "ОФЗ-ПД 26226",
        "coupon_type": "fixed",
        "analysis_group": "short_fixed_selected",
    },
    {
        "secid": "SU26207RMFS9",
        "issue_name": "ОФЗ-ПД 26207",
        "coupon_type": "fixed",
        "analysis_group": "short_fixed_selected",
    },

    # Среднесрочные ОФЗ-ПД
    {
        "secid": "SU26232RMFS7",
        "issue_name": "ОФЗ-ПД 26232",
        "coupon_type": "fixed",
        "analysis_group": "medium_fixed_selected",
    },
    {
        "secid": "SU26236RMFS8",
        "issue_name": "ОФЗ-ПД 26236",
        "coupon_type": "fixed",
        "analysis_group": "medium_fixed_selected",
    },
    {
        "secid": "SU26242RMFS6",
        "issue_name": "ОФЗ-ПД 26242",
        "coupon_type": "fixed",
        "analysis_group": "medium_fixed_selected",
    },

    # Долгосрочные ОФЗ-ПД
    {
        "secid": "SU26240RMFS0",
        "issue_name": "ОФЗ-ПД 26240",
        "coupon_type": "fixed",
        "analysis_group": "long_fixed_selected",
    },
    {
        "secid": "SU26243RMFS4",
        "issue_name": "ОФЗ-ПД 26243",
        "coupon_type": "fixed",
        "analysis_group": "long_fixed_selected",
    },
    {
        "secid": "SU26238RMFS4",
        "issue_name": "ОФЗ-ПД 26238",
        "coupon_type": "fixed",
        "analysis_group": "long_fixed_selected",
    },

    # ОФЗ-ПК с плавающим купоном
    {
        "secid": "SU29024RMFS5",
        "issue_name": "ОФЗ-ПК 29024",
        "coupon_type": "floating",
        "analysis_group": "floating_coupon",
    },
    {
        "secid": "SU29025RMFS2",
        "issue_name": "ОФЗ-ПК 29025",
        "coupon_type": "floating",
        "analysis_group": "floating_coupon",
    },
    {
        "secid": "SU29026RMFS0",
        "issue_name": "ОФЗ-ПК 29026",
        "coupon_type": "floating",
        "analysis_group": "floating_coupon",
    },
]


## 3. Вспомогательные функции для работы с ISS API


In [4]:

def request_json(url: str, params: dict | None = None, timeout: int = 60, retries: int = 5, pause: float = 2.0):
    """
    Запрос к ISS API с повторными попытками.
    Возвращает JSON-ответ или None, если все попытки закончились ошибкой.
    """
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            if response.status_code == 200:
                return response.json()
            last_error = f"HTTP {response.status_code}"
        except Exception as e:
            last_error = e

        print(f"[RETRY {attempt}/{retries}] {url}")
        print("Params:", params)
        print("Error:", last_error)
        time.sleep(pause * attempt)

    print(f"[FAILED AFTER RETRIES] {url}")
    print("Params:", params)
    print("Last error:", last_error)
    return None


def block_to_df(data: dict, block_name: str) -> pd.DataFrame:
    """
    Преобразует блок ISS JSON в pandas DataFrame.
    """
    if not isinstance(data, dict):
        return pd.DataFrame()

    block = data.get(block_name)
    if not isinstance(block, dict):
        return pd.DataFrame()

    columns = block.get("columns", [])
    rows = block.get("data", [])

    if not columns:
        return pd.DataFrame()

    return pd.DataFrame(rows, columns=columns)


def load_paginated_history(secid: str, start_date: str, end_date: str, board: str = "TQOB", limit: int = 100) -> pd.DataFrame:
    """
    Постранично загружает исторические данные по облигации с МосБиржи.
    """
    url = (
        f"https://iss.moex.com/iss/history/engines/stock/markets/bonds/"
        f"boards/{board}/securities/{secid}.json"
    )

    all_parts = []
    start = 0

    while True:
        params = {
            "from": start_date,
            "till": end_date,
            "iss.meta": "off",
            "iss.only": "history",
            "start": start,
            "limit": limit,
        }

        data = request_json(url, params=params)
        if data is None:
            break
        df_part = block_to_df(data, "history")
        if df_part.empty:
            break
        all_parts.append(df_part)
        start += len(df_part)
        time.sleep(0.5)

    if not all_parts:
        return pd.DataFrame()

    df = pd.concat(all_parts, ignore_index=True)

    if "TRADEDATE" in df.columns:
        df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])

    return df


## 4. Отбор важных колонок и добавление признаков


In [5]:

def normalize_bond_data(df: pd.DataFrame, bond_info: dict) -> pd.DataFrame:
    """
    Оставляет важные колонки, добавляет итоговую доходность и производные признаки.
    """
    if df.empty:
        return df

    df = df.copy()

    df["ISSUE_NAME"] = bond_info["issue_name"]
    df["COUPON_TYPE"] = bond_info["coupon_type"]
    df["ANALYSIS_GROUP"] = bond_info["analysis_group"]

    if "TRADEDATE" in df.columns:
        df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])

    if "MATDATE" in df.columns:
        df["MATDATE"] = pd.to_datetime(df["MATDATE"], errors="coerce")
    else:
        df["MATDATE"] = pd.NaT

    if "YIELDATWAP" in df.columns and "YIELDCLOSE" in df.columns:
        df["YIELD"] = df["YIELDATWAP"].fillna(df["YIELDCLOSE"])
    elif "YIELDATWAP" in df.columns:
        df["YIELD"] = df["YIELDATWAP"]
    elif "YIELDCLOSE" in df.columns:
        df["YIELD"] = df["YIELDCLOSE"]
    else:
        df["YIELD"] = pd.NA

    if "MATDATE" in df.columns:
        df["YEARS_TO_MATURITY"] = (df["MATDATE"] - df["TRADEDATE"]).dt.days / 365.25
    else:
        df["YEARS_TO_MATURITY"] = pd.NA

    def maturity_group(row):
        if row["COUPON_TYPE"] == "floating":
            return "floating_coupon"

        years = row.get("YEARS_TO_MATURITY", pd.NA)
        if pd.isna(years):
            return pd.NA
        if years <= 1:
            return "short"
        if years <= 5:
            return "medium"
        return "long"

    df["MATURITY_GROUP"] = df.apply(maturity_group, axis=1)
    if "CLOSE" in df.columns:
        df["PRICE_DEV_FROM_100"] = df["CLOSE"] - 100
        df["PRICE_RETURN"] = df.groupby("SECID")["CLOSE"].pct_change()
    else:
        df["PRICE_DEV_FROM_100"] = pd.NA
        df["PRICE_RETURN"] = pd.NA

    if {"CLOSE", "FACEVALUE", "ACCINT"}.issubset(df.columns):
        df["CLEAN_PRICE_RUB"] = df["CLOSE"] / 100 * df["FACEVALUE"]
        df["DIRTY_PRICE_RUB"] = df["CLEAN_PRICE_RUB"] + df["ACCINT"]
    else:
        df["CLEAN_PRICE_RUB"] = pd.NA
        df["DIRTY_PRICE_RUB"] = pd.NA

    important_columns = [
        # Идентификаторы и даты
        "TRADEDATE",
        "SECID",
        "ISSUE_NAME",
        "SHORTNAME",
        "BOARDID",

        # Группировки для анализа
        "COUPON_TYPE",
        "ANALYSIS_GROUP",
        "MATURITY_GROUP",

        # Цены
        "OPEN",
        "HIGH",
        "LOW",
        "CLOSE",
        "WAPRICE",
        "CLEAN_PRICE_RUB",
        "DIRTY_PRICE_RUB",
        "PRICE_DEV_FROM_100",
        "PRICE_RETURN",

        # Доходности
        "YIELDATWAP",
        "YIELDCLOSE",
        "YIELD",

        # Объёмы и ликвидность
        "VOLUME",
        "VALUE",
        "NUMTRADES",

        # Купонные и облигационные характеристики
        "ACCINT",
        "FACEVALUE",
        "COUPONPERCENT",
        "COUPONVALUE",
        "MATDATE",
        "YEARS_TO_MATURITY",
        "DURATION",

        # Тип инструмента, если есть в данных МосБиржи
        "BONDTYPE",
        "BONDSUBTYPE",
    ]

    existing_columns = [col for col in important_columns if col in df.columns]
    df = df[existing_columns].copy()

    return df


## 5. Загрузка одной облигации


In [6]:

def safe_filename(text: str) -> str:
    """
    Упрощает строку для имени файла.
    """
    return (
        text.replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
            .replace("-", "-")
    )


def load_and_save_bond(bond_info: dict, start_date: str, end_date: str, out_dir: Path, board: str = "TQOB") -> pd.DataFrame:
    """
    Загружает одну ОФЗ, оставляет важные колонки и сохраняет отдельный CSV.
    """
    secid = bond_info["secid"]
    issue_name = bond_info["issue_name"]

    print("\n" + "=" * 60)
    print(f"Loading {issue_name} ({secid})")
    print("=" * 60)

    raw = load_paginated_history(
        secid=secid,
        start_date=start_date,
        end_date=end_date,
        board=board,
    )

    if raw.empty:
        print(f"[NO DATA] {issue_name} ({secid})")
        return pd.DataFrame()

    print(f"[RAW OK] shape = {raw.shape}")

    df = normalize_bond_data(raw, bond_info)

    file_name = f"{safe_filename(issue_name)}_{secid}.csv"
    file_path = out_dir / file_name

    df.to_csv(file_path, index=False, encoding="utf-8-sig")

    print(f"[SAVED] {file_path}")
    print(f"[FINAL SHAPE] {df.shape}")
    print(f"[DATE RANGE] {df['TRADEDATE'].min().date()} — {df['TRADEDATE'].max().date()}")

    if "YIELD" in df.columns:
        print(f"[YIELD COVERAGE] {df['YIELD'].notna().mean():.1%}")

    return df


## 6. Загрузка всех выбранных выпусков


In [7]:

all_bonds_data = []

for bond_info in BONDS:
    df_bond = load_and_save_bond(
        bond_info=bond_info,
        start_date=START_DATE,
        end_date=END_DATE,
        out_dir=OUT_DIR,
        board=BOARD,
    )

    if not df_bond.empty:
        all_bonds_data.append(df_bond)

if not all_bonds_data:
    raise RuntimeError("Не удалось загрузить данные ни по одной облигации.")

df_all = pd.concat(all_bonds_data, ignore_index=True)
df_all = df_all.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

combined_path = OUT_DIR / "ofz_all_combined_2016_2026.csv"
df_all.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("Combined dataset:", combined_path)
print("Shape:", df_all.shape)

display(df_all.head())



Loading ОФЗ-ПД 26219 (SU26219RMFS4)
[RAW OK] shape = (2488, 48)
[SAVED] /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/ofz_moex_csv_2016_2026/ОФЗ-ПД_26219_SU26219RMFS4.csv
[FINAL SHAPE] (2488, 32)
[DATE RANGE] 2016-07-06 — 2026-04-30
[YIELD COVERAGE] 100.0%

Loading ОФЗ-ПД 26226 (SU26226RMFS9)
[RETRY 1/5] https://iss.moex.com/iss/history/engines/stock/markets/bonds/boards/TQOB/securities/SU26226RMFS9.json
Params: {'from': '2016-05-03', 'till': '2026-05-03', 'iss.meta': 'off', 'iss.only': 'history', 'start': 900, 'limit': 100}
Error: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
[RAW OK] shape = (1832, 48)
[SAVED] /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/ofz_moex_csv_2016_2026/ОФЗ-ПД_26226_SU26226RMFS9.csv
[FINAL SHAPE] (1832, 32)
[DATE RANGE] 2019-02-06 — 2026-04-30
[YIELD COVERAGE] 100.0%

Loading ОФЗ-ПД 26207 (SU26207RMFS9)
[RAW OK] shape = (2531, 48)
[SAVED] /Users/daniilsest

,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,BONDTYPE,BONDSUBTYPE
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,96.0501,96.0501,95.2501,95.5993,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,None,None,2027-02-03,10.751540,2610,None,None
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.6000,95.8000,95.3550,95.6201,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,None,None,2027-02-03,10.748802,2610,None,None
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8888,95.9000,95.7000,95.9000,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,None,None,2027-02-03,10.746064,2610,None,None
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.8000,95.9500,95.2900,95.7000,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,None,None,2027-02-03,10.735113,2606,None,None
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,95.9499,96.5999,95.7600,96.4500,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,None,None,2027-02-03,10.732375,2610,None,None


## 7. Контроль качества загрузки


In [8]:

summary = (
    df_all
    .groupby(["SECID", "ISSUE_NAME", "COUPON_TYPE", "ANALYSIS_GROUP"], as_index=False)
    .agg(
        date_min=("TRADEDATE", "min"),
        date_max=("TRADEDATE", "max"),
        rows=("TRADEDATE", "count"),
        close_na=("CLOSE", lambda x: x.isna().sum()),
        yield_na=("YIELD", lambda x: x.isna().sum()),
        yield_coverage=("YIELD", lambda x: x.notna().mean()),
        close_mean=("CLOSE", "mean"),
        close_min=("CLOSE", "min"),
        close_max=("CLOSE", "max"),
        yield_mean=("YIELD", "mean"),
        yield_min=("YIELD", "min"),
        yield_max=("YIELD", "max"),
        volume_mean=("VOLUME", "mean"),
        value_mean=("VALUE", "mean"),
    )
)

summary_path = OUT_DIR / "ofz_download_summary_2016_2026.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Summary saved to:", summary_path)
display(summary)


Summary saved to: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/ofz_moex_csv_2016_2026/ofz_download_summary_2016_2026.csv


,SECID,ISSUE_NAME,COUPON_TYPE,ANALYSIS_GROUP,date_min,date_max,rows,close_na,yield_na,yield_coverage,close_mean,close_min,close_max,yield_mean,yield_min,yield_max,volume_mean,value_mean
0,SU26207RMFS9,ОФЗ-ПД 26207,fixed,short_fixed_selected,2016-05-04,2026-04-30,2531,15,0,1.000000,99.622430,79.023,117.988,9.731351,5.04,21.67,4.206808e+05,4.192155e+08
1,SU26219RMFS4,ОФЗ-ПД 26219,fixed,short_fixed_selected,2016-07-06,2026-04-30,2488,16,0,1.000000,98.439252,78.700,114.951,9.747452,5.0,21.88,2.740004e+05,2.677882e+08
2,SU26226RMFS9,ОФЗ-ПД 26226,fixed,short_fixed_selected,2019-02-06,2026-04-30,1832,15,0,1.000000,98.574169,78.900,116.166,10.396015,4.99,21.83,3.155227e+05,3.104577e+08
3,SU26232RMFS7,ОФЗ-ПД 26232,fixed,medium_fixed_selected,2019-12-11,2026-04-30,1617,15,0,1.000000,88.611918,68.800,105.150,10.848182,5.18,20.95,2.570138e+05,2.333958e+08
4,SU26236RMFS8,ОФЗ-ПД 26236,fixed,medium_fixed_selected,2020-11-25,2026-04-30,1379,15,0,1.000000,83.100337,65.850,100.625,11.733365,5.66,20.53,3.076787e+05,2.627519e+08
5,SU26238RMFS4,ОФЗ-ПД 26238,fixed,long_fixed_selected,2021-06-23,2026-04-30,1234,15,0,1.000000,68.222588,48.528,100.998,12.107472,7.19,16.65,1.194103e+06,7.182227e+08
6,SU26240RMFS0,ОФЗ-ПД 26240,fixed,long_fixed_selected,2021-06-30,2026-04-30,1229,26,11,0.991050,70.238069,50.584,99.890,12.356281,7.16,17.36,4.552370e+05,3.149111e+08
7,SU26242RMFS6,ОФЗ-ПД 26242,fixed,medium_fixed_selected,2023-01-25,2026-04-30,830,1,1,0.998795,85.535952,69.239,100.000,13.880072,9.17,20.01,3.679761e+05,3.124457e+08
8,SU26243RMFS4,ОФЗ-ПД 26243,fixed,long_fixed_selected,2023-06-21,2026-04-30,730,1,1,0.998630,76.155753,62.600,101.299,14.402428,9.86,17.65,9.056685e+05,6.670708e+08
9,SU29024RMFS5,ОФЗ-ПК 29024,floating,floating_coupon,2023-05-03,2026-04-30,763,1,1,0.998689,96.120803,91.829,99.069,0.0,0.0,0.0,8.764126e+04,8.466628e+07
